In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from scipy.io import loadmat
import os, warnings

: 

In [ ]:
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
image_path = Path("data/samples/IMG_119.jpg")

image_bgr = cv2.imread(str(image_path))

if image_bgr is None:
    raise FileNotFoundError(f"Image not found: {image_path}")

print("Loaded image shape:", image_bgr.shape)

In [ ]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.title("Original Image (RGB)")
plt.axis("off")
plt.show()

In [ ]:
resized = cv2.resize(image_rgb, (768, 480))

plt.figure(figsize=(10, 6))
plt.imshow(resized)
plt.title("Resized Image (640x480)")
plt.axis("off")
plt.show()
print(resized.shape)

In [ ]:
blurred = cv2.GaussianBlur(resized, (3, 3), 0,5)

plt.figure(figsize=(10, 6))
plt.imshow(blurred)
plt.title("Gaussian Blurred Image")
plt.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].imshow(image_rgb)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(resized)
axes[1].set_title("Resized")
axes[1].axis("off")

axes[2].imshow(blurred)
axes[2].set_title("Gaussian Blur")
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
mat_path = Path("data/annotations/GT_IMG_119.mat")
mat = loadmat(mat_path)

print(type(mat))
print(mat.keys())

In [ ]:
locations = mat["image_info"][0, 0][0, 0][0]
print(type(locations))
print(locations.shape)
print(locations[:5])

In [ ]:
scale_x = 768/1024
scale_y = 480/768

scaled_locations = locations.copy()
scaled_locations[:, 0] *= scale_x
scaled_locations[:, 1] *= scale_y

print("Scaled locations:")
print(scaled_locations[:5])

In [ ]:
original_vis = image_rgb.copy()

for x, y in locations.astype(int):
    cv2.circle(original_vis, (x, y), radius=4, color=(255, 0, 0), thickness=-1)

plt.figure(figsize=(12, 8))
plt.imshow(original_vis)
plt.title("Resized Image with Scaled Points")
plt.axis("off")
plt.show()

In [ ]:
resized_vis = resized.copy()

for x, y in scaled_locations.astype(int):
    cv2.circle(resized_vis, (x, y), radius=4, color=(255, 0, 0), thickness=-1)

plt.figure(figsize=(10, 7))
plt.imshow(resized_vis)
plt.title("Resized Image with Scaled Points")
plt.axis("off")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(original_vis)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(resized_vis)
axes[1].set_title("Resized + Scaled Points")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def resize_image_and_scale_points(image_rgb, points, new_size):
    orig_h, orig_w = image_rgb.shape[:2]
    new_w, new_h = new_size

    resized_image = cv2.resize(image_rgb, (new_w, new_h))

    scale_x = new_w / orig_w
    scale_y = new_h / orig_h

    scaled_points = points.astype(np.float32).copy()
    scaled_points[:, 0] *= scale_x
    scaled_points[:, 1] *= scale_y

    return resized_image, scaled_points, scale_x, scale_y

In [ ]:
resized_image2, scaled_points2, sx, sy = resize_image_and_scale_points(
    image_rgb, points, (640, 480)
)

print(sx, sy)
print(scaled_points2[:3])